# Causal grouped-query attention with padding and a sliding window

A decoder self-attention layer as used in modern LLMs: grouped-query heads, a causal mask, right-padded sequences of different lengths, and optionally a sliding window that limits how far back each token looks.

Write `causal_gqa(x, Wq, Wk, Wv, Wo, n_heads, n_kv_heads, lengths, window=None)` in PyTorch.

## Shapes
- `x`: `float64` `(B, T, D)`. $H$ = `n_heads` divides $D$; head width $d_h = D / H$. $H_{kv}$ = `n_kv_heads` divides $H$.
- `Wq`: `(D, H d_h)`, `Wk`, `Wv`: `(D, H_kv d_h)`, `Wo`: `(H d_h, D)`. No biases. Projections are `x @ W`.
- `lengths`: `int64` `(B,)`, $1 \le \ell_b \le T$: sequence $b$ has real tokens at positions $0, \dots, \ell_b - 1$; the rest is padding.
- `window`: `None` or an int $w \ge 1$.

## Computation
1. $Q = xW_q$, $K = xW_k$, $V = xW_v$, split into heads: feature block $h$ (columns $h d_h, \dots, (h+1)d_h - 1$) is head $h$. Query head $h$ uses key/value head $\lfloor h / (H/H_{kv}) \rfloor$.
2. Scores $S_{b,h,i,j} = Q_{b,h,i} \cdot K_{b,g(h),j} / \sqrt{d_h}$.
3. Query $i$ may attend to key $j$ only if $j \le i$, $j < \ell_b$, and (if `window` is set) $i - j < w$. Disallowed scores are excluded from the softmax.
4. Softmax over $j$, weighted sum of $V$, concatenate heads in order, multiply by `Wo`.
5. Rows of padding queries ($i \ge \ell_b$) are set to **exactly zero** in the output.

## Output
`float64` tensor `(B, T, D)`. Use tensor ops only (no Python loops over batch, heads or positions).

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import math
import torch

def causal_gqa(x, Wq, Wk, Wv, Wo, n_heads, n_kv_heads, lengths, window=None):
    """Return the (B, T, D) output of causal GQA self-attention."""
    B, T, D = x.shape
    dh = D // n_heads
    q = (x @ Wq).view(B, T, n_heads, dh).transpose(1, 2)
    k = (x @ Wk).view(B, T, n_kv_heads, dh).transpose(1, 2)
    v = (x @ Wv).view(B, T, n_kv_heads, dh).transpose(1, 2)
    k = k.repeat(1, n_heads // n_kv_heads, 1, 1)      # TODO: wrong head order, and no masks at all
    v = v.repeat(1, n_heads // n_kv_heads, 1, 1)
    p = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(dh), dim=-1)
    return (p @ v).transpose(1, 2).reshape(B, T, D) @ Wo


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 1, 1, {"__nd__": [3], "dtype": "int64"}], "e": {"__nd__": [[[1.0, 0.0], [0.33023845067334306, 0.6697615493266569], [0.7517449217422769, 0.7517449217422769]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, 1, 1, {"__nd__": [2], "dtype": "int64"}, 1], "e": {"__nd__": [[[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[-1.738266398496882, -1.3366427931811324, -1.361106708564987, -0.35161713127840977, -2.3125815796967033, -0.18889719608460778, -0.957229228096346, 0.8936001849299788], [0.956847237579234, 1.3922582291390866, 0.7674701130947078, -0.053029778757267734, 0.8597939889439096, 1.5054811563838433, -0.6535945334170485, 0.610351145830187], [-0.042673827710852374, 1.4400167254152394, -0.8368950200968434, -0.3015466095655266, 0.36233859316943917, 0.25811026702099754, -1.6394479624476481, 0.360155232237386], [-0.11849769951697288, -0.23974784920156203, -0.15530166200229314, 0.21897170507821917, -1.8163956614546406, 1.5524665679968663, -0.8614416732885682, -2.2413678581872873], [-0.08197449383484104, 1.4574804175244145, -0.5186009711032398, 1.5512756209056682, 1.5569420010285768, -0.8627319171113763, -2.4651208152773547, -1.2351827568078488]], [[1.1874322225543146, -0.8167721736209068, -1.5106774902505407, -1.3376946740539473, 0.00018014422029518804, -0.026108483729201715, 0.8720449149097206, 0.9890499704668804], [-0.9322490224957446, -0.15675869871669845, -1.1339346465649662, 0.07189067206172675, -1.1506078219780655, -1.200251555452681, 2.1234219339526375, 0.032388888425520096], [0.6431786123703356, 2.5380983786528484, 0.7858388236905651, -0.11414260193201425, 0.05458995154441833, -0.7444919671299762, 1.094338440936761, -0.3302786854546604], [1.7627455255994957, -0.3447585414426272, -0.3536229319237997, -0.41937740024787623, 0.6256938797342935, 0.18693606942873356, 1.3714324073778508, -0.5533496226261118], [0.29023814454305935, 1.9379617345617723, -0.24038168695409634, 0.5326127940361377, 1.5718822401651174, 0.5534457799103193, -0.6345051681060726, -2.062350018165741]]], "dtype": "float64"}, {"__nd__": [[0.9572654577999722, 0.17212571285214617, 0.5042115550428127, -0.19052541965532305, 0.7273926892599234, 0.2078157440772202, 0.019918977325759576, -0.36072687648929513], [0.14727388823264265, -0.19461201297241704, -0.08910767174831313, -0.6490043269569131, -0.19622157209499178, 0.06395161393519193, -0.5003224640012388, 0.029558603826681823], [0.2653175515831378, -0.12270397627884765, -0.2483136639838414, -0.24804967504384542, 0.37645692239233397, -0.6138020181195643, -0.12018055732423011, -0.16776204299401426], [0.05759881698373313, -0.027828034792184504, -0.4430024626496387, 0.8160357948681904, 0.07104470783516635, -0.5368153307898885, -0.3565733799346656, 0.0390339502018002], [0.2451879753469839, -0.5368620258327569, 0.2845347496403827, -0.5493168331641395, -0.3177555354814515, 0.3412405050272786, 0.6803988917178625, -0.03388314864870168], [-0.1238154469622863, 0.5071596594558052, -0.42879155477200004, 0.11116873228719702, 0.09329822886283062, 0.40372870408428296, 0.11943137493322105, -0.10655773375902547], [0.24582183375460753, -0.4506259446520817, 0.32563057251344524, -0.26907060368023966, -0.364146360421991, -0.14705855603644138, 0.01840057661108232, 0.28423436421469644], [-0.08506826949937707, 0.29428914058739764, 0.39003299443286876, -0.04230352317760859, -0.39261884421216564, 0.36790043568953634, 0.11017929893876921, -0.8460774141023809]], "dtype": "float64"}, {"__nd__": [[-0.39531546078552626, 0.057264676091296124, -0.5487622057194309, -0.2725680569840757], [-0.6851688563058101, 0.16090544489112535, 0.09660712270530362, 0.07045024055142646], [0.5789896252281995, -0.14104066832314913, -0.11567207417958864, 0.8079693128293421], [-0.5203451893215926, 0.2916651613804826, -0.5260386136646457, 0.0945023156891136], [0.8463130500203282, 0.10854240996605254, -0.3087291088016546, 0.9536864236370617], [-0.21752854988603432, -0.08247859993474545, 0.12174882408974548, -0.5370260790119606], [0.44603931353839316, 0.6475300235695804, 0.3035063259195313, 0.1321444107491229], [-0.05971907521457731, 0.3546942328891649, -1.260031312076202, -0.049908104879552315]], "dtype": "float64"}, {"__nd__": [[0.10245029194232685, -0.7942120774698959, 0.1704669352356163, -0.05100199385851402], [-0.01382638865472908, 0.04055649350491937, 0.2300548057322147, 0.18318501692046585], [-0.4149407942557208, -0.04678479081593365, -0.07283933002308066, 0.015368975396096347], [-0.23643651620603068, 0.3879415649218858, 0.5218193466664264, 0.608679913185924], [0.7622609098791048, 0.597312262061969, -0.18397265332594204, -0.24575074902257465], [-0.15905557422622008, 0.22337541759195423, -0.05193225779977215, 0.3964374306190318], [0.6601565955148356, -0.17210449586073567, 0.3918830906951166, -0.157177915028238], [-0.5065990006985447, -0.5075374132555538, -0.12765185912173024, -0.7596403767539005]], "dtype": "float64"}, {"__nd__": [[0.19972419339306893, 0.6042724017876422, -0.46602484305820924, -0.1337256369077761, 0.6174233240074871, 0.13889279855624154, 0.07826664989747774, -0.10461359451993468], [0.4074635177954218, 0.1316969330739027, -0.4336750888390863, -0.9728373208140436, -1.0726769190473486, 0.26306857902395303, -0.10643379175990955, 0.5792519144509665], [0.20399781316934243, -0.5088147981094769, 0.11615557325877453, 0.4767446384051633, -0.3607026750625224, -0.20349694664372323, 0.6963948709844473, -0.6985568032114899], [-0.4361147653384936, 0.420015946853958, 0.07959829428277854, -0.4569963416776962, -0.5820981645420474, 0.1853819016607482, 0.4857078123295291, 0.3306338050085783], [0.16447412990102292, 0.5981892942472067, 0.12445711927287946, -0.33299089807888066, 0.8127521553003542, -0.49362164103626843, -0.053815407306600484, -0.02922692644921306], [0.572454515191564, -0.30548129607286, -0.11068804879167747, -0.7121625667335949, 0.13501392174335528, 0.17806499493709055, 0.2602709200226556, -0.370693534168737], [0.09454504185970104, -0.35365793836962944, -0.1916475444852611, -0.031431975749438436, 0.5170585869047722, -0.17611256272635634, 0.0695891582732151, -0.33897369799995275], [-0.22931376778448037, -0.28207533755407327, -0.06521927425703855, 0.3047482406195365, -0.4445210737475721, 0.28509352645033864, -0.3835742577526713, 0.13080300558080632]], "dtype": "float64"}, 4, 2, {"__nd__": [5, 3], "dtype": "int64"}], "e": {"__nd__": [[[-1.265563308504364, -0.40652545306791615, 1.102090854333462, 0.3009812948702253, -0.6938262129407765, 0.2444246649864189, -1.937374302063141, 1.8287039903204272], [-0.47243800814688974, 0.2338657370061817, 0.5006504242784622, -0.422811940179044, 0.004171069112167358, 0.24251016616534699, -1.3206619584293235, 1.2697387612885884], [-0.434008467397609, -0.015251489859431634, 0.46269649299976423, -0.4553892233254163, -0.5907654721164037, 0.2921162395324018, -0.8913200704639934, 1.0774277529641658], [1.1425138372567814, -0.7295417683982167, 0.0860465792665509, -2.5892569926685423, -0.11521970662670891, 0.6236940213856881, -0.05581283452830321, -0.02694480008218403], [0.2764136747788527, -0.4446592560890789, -0.16015995304584568, -2.1047977268910323, -1.7741455868321652, 0.9808405527934909, -0.3616844526427528, 0.9589633824636352]], [[-0.2343233480981236, 0.01785350547219109, 0.6995353760279024, 4.296439896705479, 3.8702944676104583, -1.6710256155803505, 0.33935489637496774, -2.2240172887232657], [-0.394642571194441, 0.1239809759690387, 0.36189116993655396, 3.3548238738687997, 2.7848593239520802, -1.2953809630484527, 0.47968678985827196, -1.6752656915848714], [0.07878571816355912, 0.05467816874910439, 0.35923200663068156, 2.9600832533500614, 3.4706939778883497, -1.4533833017833468, 0.36413707725067446, -2.0047858200796194], [0.0, -0.0, 0.0, 0.0, 0.0, -0.0, 0.0, -0.0], [0.0, -0.0, 0.0, 0.0, 0.0, -0.0, 0.0, -0.0]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[-0.560870965666268, -0.31224180271226776, -0.4021692347232762, 1.0870613289026454, 0.2394661460452593, 1.0559872773133139, -0.38340030439777556, -0.2582603578332374], [1.5923687082932494, -1.6704613982986534, -0.9014681893014775, -0.2014796354504079, 1.7977227001980955, -0.633399520432185, 0.7699411198902094, 1.5583218731168709], [-0.2608745394559887, -0.19837509448756974, 1.6461708345876043, -0.5787852279754228, -1.0786494930343222, 0.8029630260193307, -0.5418881870963893, 1.2721842522288556], [0.5135474956511674, -1.7799098873321286, 0.25332469913849437, -0.014213846692776116, -0.6893300001570001, 1.0785818356114014, -1.797825175632008, -1.2800208576174648], [-0.5234266949164023, 0.7662141376599587, -1.5694801544923136, -0.6661272213169701, -0.7451257621496428, -1.5424983132721142, -0.41986233875211537, -1.18571085332387]], [[-0.4889876495432731, 0.2486882945988942, -0.9534118342920758, -0.9142826632962725, -1.0316511638293742, -0.8933054947581841, -1.6148024657827056, -1.2263999142508761], [0.07719191957678519, -0.9243544583504244, 3.2339018997770634, -0.8600130621304797, 0.4160232040597361, -1.5521503681442799, -0.31256815258660375, -0.5912597525566127], [-0.12163392660830079, 1.2945406371939017, -0.20827333894827663, -1.2501996254287515, 0.08347004433274519, -1.0038149259644478, -0.15295570559470914, 1.5730545295598348], [0.9302760443668044, 1.0671787309950316, 0.5212999120526557, -0.031584323505555144, -1.55792171748771, -0.4846561933752778, -0.5406911118615139, 0.6760981800274841], [0.9945112303009566, 0.4785252373086558, -0.7995845410759872, 1.8504661621271392, 1.4198445308034204, -0.217647563808852, 0.19711457610825392, 0.6402798930741478]]], "dtype": "float64"}, {"__nd__": [[0.04155653328435752, -0.039414235752022325, 0.1977679033576748, 0.004162437563774603, 0.08172454474133357, -0.644540061893232, -0.6430357169685029, 0.6964346849845663], [-0.6089477547752808, 0.19489153086904454, 0.12815078775564764, 0.0890035033671494, -0.27744189128382285, 0.40631108042097747, 0.14040725399134144, -0.13686445170831882], [-0.5628853399541319, -0.0918830260288467, -0.2395825874146964, -0.5033272415618557, 0.15244821320094704, 0.23313125320182407, 0.3341825297262298, 0.04968410773011865], [-0.09534484503847376, 0.45963634634259165, 0.3133112990238553, 0.9282996318723241, 0.23085932248556157, -0.3671186922554714, 0.39538527779247107, 0.9428106927239658], [0.1197510760233162, -0.062440164748131854, -0.2274856043521758, 0.7773301119744098, -0.061493177799418236, -0.08450380626279541, 0.13672419635527164, -0.554299367810782], [-0.09321849108107626, -0.4913596632882731, -0.039768079937019356, 0.21704169057768802, 0.3896997824032966, -0.14871682127561456, 0.3484287220653545, -0.13738252810956447], [-0.6428102644100955, 0.2503520794061215, -0.06145002842314489, 0.06589756150302303, -1.0276695211979234, 0.10941792128180805, 0.12443673794678847, 0.12457528554198234], [0.44358597572452796, -0.11732813386035156, 0.8894895930558789, 0.3212413797067418, 0.24960248087383943, -0.9685931055362815, -0.7572435734143812, 0.6337821299704391]], "dtype": "float64"}, {"__nd__": [[0.19057727148550105, 0.40804465153057845, -0.1162833513695033, -0.010815922240981449], [0.27405910970161157, -0.2769064963836297, 0.5716875055670618, -0.4017677497123721], [0.13271778231993703, -0.06276297521647455, 0.18929763967260094, 0.5485022599008207], [-0.3984163666309407, 0.2337670849062548, 0.10185854530151964, 0.3920597567925472], [0.04598279269140592, -0.14078705688253512, 0.2577734961627032, 0.5912802864156811], [0.3851200002954445, 0.12126308750039866, 0.2295484946808346, -0.557510075940242], [-0.15669558106530584, -0.04101937798450503, -0.15482380846378158, -0.22023539821564744], [0.04109119600484437, 0.22054210778215105, 0.836224883746191, -0.22050393721167785]], "dtype": "float64"}, {"__nd__": [[-0.3038980308255937, 0.45845647029181946, 0.12182960646350767, -0.2959093043431613], [0.6722313566890072, -0.024033007820566957, -0.07141702449810441, -0.18703181194024726], [-0.3102738921897042, 0.09809349550714429, -0.32584579735827207, 1.0386542366540659], [0.18886282766080087, -0.04493706311415848, -0.25642511657682826, 0.15407687947365656], [0.44854813885784356, 0.14547111686745737, -0.40911498383641653, -0.045648607240235284], [-0.6363148379662432, 0.6185397117986153, 0.3174063767798955, 0.12682156590124563], [0.16058709594529064, -0.18354720631304708, -0.014780091145439697, 0.42800166997854483], [0.02596462758224696, 0.27911197592299136, 0.13168815666329434, 0.1365334388803548]], "dtype": "float64"}, {"__nd__": [[0.17139334095107236, 0.4603225543338148, -0.072005520176789, 0.3167932344030555, -0.2707633507573194, -0.18674145831687472, -0.24038514026653401, 0.38777836550915445], [-0.07357058397960778, 0.5381330438447844, -0.3597433256335098, 0.20273457965725328, -0.41325942135423155, 0.49826564415708935, -0.22770147342884434, 0.23205617022215685], [-0.22544024268304907, -0.25643909100662704, 0.1461174279961673, -0.6366566000191106, -0.4274707026833971, 0.6536789571080935, -0.27911554892074164, 0.37532309571636224], [-0.29493037920367315, -0.6070218174763145, -0.11645291186151695, 0.3948198809645498, 0.038110914390671616, -0.06986932573547826, 0.3899057708581605, -0.25613492992876286], [0.33643523385709934, 0.17177290771952403, -0.3643489156561662, 0.41545309294681226, 0.49407758587226447, -0.052949695721288007, 0.09179665857396678, -0.06862634956290399], [-0.4058864437450034, 0.35851167960463526, -0.3618562641220622, 0.2512000136394252, -0.17260359683004672, -0.3713822430381797, 0.2843310424171191, 0.13202642487107424], [0.7764317795610446, 0.010051554740587882, 0.09796596398466104, 0.08450995829167302, -0.061506563237515605, 0.5650437070697645, 0.6402022804528198, 0.31125365061657256], [-0.3635350663463188, 0.22061395816688725, 0.06413816252207942, -0.1386092384136695, -0.5790497966920571, 0.3951209153637688, 0.5215510566625688, 0.1663396626134933]], "dtype": "float64"}, 4, 2, {"__nd__": [5, 3], "dtype": "int64"}, 2], "e": {"__nd__": [[[-0.014389521706745727, -0.15021020127227086, -0.1648441009321573, 0.3136669809420335, 0.19160534605428114, -0.00521179856045198, 0.1629552471380287, -0.29632518974886723], [-0.05036702339933524, -0.3920918640883916, -0.261256953830494, 0.41534846380300455, 0.22954865404451616, 0.03156910340736514, -0.12395952228886858, -0.2926829780864084], [-0.5735352326098426, 0.3701537355072408, -1.1932060837584417, 1.6053926739870192, 0.5425905759456587, -0.8854686046725468, 1.2745535610837861, -0.807248608856336], [-0.8448849453104531, 0.6163433921200322, -1.2833553260114836, 1.8013157062480294, 0.29393094271424625, -0.606505064861231, 2.831129533796825, -1.0645021275058935], [1.1996299258707117, -0.5991509744404624, 0.2407826387539574, -1.106039482216011, 1.3437577560033136, 0.6639705790212989, -1.1660424621978958, -0.2823209674014601]], [[2.3941545175531673, -0.9258522085316959, 0.9678090351793485, -0.6833281412535048, 1.884129235035573, -0.12986250670958616, -1.533555320473826, -0.2521796549306908], [-0.6110710357346011, 0.4610604997946252, -0.18039259827335152, -0.1216535100134365, 0.05015834280159459, -1.8071187569738418, 0.05919547002462387, 0.17786316356146767], [-2.5167264850383786, 0.8711789505931492, 0.11729759868952597, -0.9220418684183211, -2.1179459354263734, 0.06116925585828781, 0.41258429192058754, 0.5898097588078381], [0.0, -0.0, 0.0, -0.0, -0.0, 0.0, -0.0, 0.0], [-0.0, 0.0, 0.0, -0.0, -0.0, -0.0, -0.0, 0.0]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[0.5601368924456227, 1.862743281863712, 2.0399672551597146, 0.3273084628323444, -2.747089437745501, 0.8175570323688326, 0.25750309060281473, 0.7897225978067951, -0.4121878307269395, -0.8969552717173769, 0.6126949182097514, -0.19669063697009215], [0.20412576542082475, 0.516590019953213, 0.20131868803598238, -1.3402438573805668, 0.6684731642730136, -0.3956027116479189, -0.12074411380212392, 0.5113555619913495, -1.3665442623529966, -0.1835070473232754, -1.746786167660629, -0.48186153549591565], [0.4096670165022785, -0.07690535285141395, -0.22228245743973113, 0.6266839127331526, -0.35706578692282176, -1.554538757868063, 1.0958158184845588, 0.23265429419996583, 0.8052873236132189, -0.3945046192222055, -0.4247748842381474, -0.737142668608095], [-1.9241280283631934, -0.4742523016564395, -0.6128571585521315, 1.560513913775564, -1.9220071220497779, 0.0030023169201341036, -0.21344540641074353, -0.11847500375215371, -0.1971306511457009, 0.1671822330732233, 0.7973518582959713, -1.7093943772358013], [0.7682654100283441, -0.34877129355504644, -1.0580937671561363, -1.9339701116813734, -1.9043972649962706, -0.16302333062426594, -1.2126497105957184, -1.218423766469989, -0.5895909645665949, -0.07720280129943148, -0.566559986364636, -1.2274507037973599], [-0.7198516737955624, -0.26789451790268737, -1.139807383085538, 0.8141037875290403, 1.18148488509045, 1.2072910800402712, -0.40028591408044734, -0.4172771824379436, -0.9886246820752228, 1.3025071945938607, -0.5436244074760348, 2.1045088195931703]]], "dtype": "float64"}, {"__nd__": [[-0.8394059854070767, 0.06492932702894887, 0.17622381394326087, -0.021264509114810138, 0.21056756847063213, 0.7746333157583627, -0.03725553220288677, 0.30861394989971114, -0.19751454968494725, -0.5176457438908976, -0.43929326436880906, 0.3179530102085178], [0.11148332320423218, -0.5875745067233051, 0.11092867218215469, 0.22737021442658403, 0.18216930894574565, 0.439187635139297, -0.5903781315938332, -0.38519505153170613, -0.43513891038276403, 0.05119727531625602, 0.5152538119957835, -0.1801247207559052], [0.13345374870713497, 0.34369346408330387, -0.185048493797975, 0.32055272333332535, -0.04721038128464305, 0.7051608619967501, 0.5598366556085635, -0.17578980198513813, -0.2096937542069952, 0.7406898269129487, -0.34012851987487247, 0.46935771274874016], [-0.15566887814636968, -0.4523736768725896, 0.6986601620440347, 0.18403181967488966, -0.40198637439505114, -0.08370554170638154, -0.2810818509647106, -0.3744010964542107, -0.030744369103229953, 0.44756847224783913, -0.5528505871254431, 0.2090778463383509], [-0.5696431659118637, -0.018595022250691696, -0.5679051790901086, -0.526259067841639, -0.26949068789574754, 0.39877238320624275, 0.33328486943321073, -0.056039870136833675, -0.12893803197257, 0.04387391696959359, 1.1227844826245468, 0.13597580576009277], [-0.055043955637087943, -0.19603372153685122, -0.12754710230560826, 0.24427536213216447, 0.22764916977561112, -0.9574521352775278, 0.8358155244228556, 0.17341536884160327, -0.041504863336069925, 1.0674972438255332, -0.6214955552951171, -0.5034499077131888], [0.4439288100301861, -0.002734180749772299, 0.35075006082397764, 0.4876698652963827, 0.7281597233102095, 0.2637382256539908, -0.25701563174385605, -0.6728075533476636, 0.051884716650613574, -0.7061093661659864, 0.02142423020863905, -0.32217199473050007], [0.1501467859130156, 0.27546001758376526, -0.01949406857736355, -0.5545807478286663, 0.05601329491465158, -0.2931591968573948, 0.12103217510346374, -0.18669102219185796, 0.9312391850301094, -0.2232924252607556, -0.7759738663607845, -0.37006325678399404], [0.02911744409614745, 1.1098983802701798, 0.5582751098265125, -0.24192521215571694, -0.0015444030362524288, -0.48714925630839245, -0.5911029636842583, 0.3729707607567852, 0.16140604115512405, -0.30727293947657747, -0.29315561419938063, 0.16654404659000907], [0.2677539138378725, -0.3278420624959648, -0.693533735487827, 0.35640689732654973, -0.30067510314897555, -0.7407302645626315, -0.08142059506037251, 0.0801562906057991, 0.38196421386545165, -0.17997608607967427, 0.1068552274546567, -0.5151082726787396], [0.18265390874441143, -0.625369065472817, 0.23935101489176433, 0.06324841839718542, -0.6174309729709693, -0.25810145981094673, 0.6625389748642472, -0.4311801419010909, -0.6684095506598757, -0.12131511637635567, 0.06166192178710701, 0.32428297153462293], [-0.06382777172566778, -0.19209378055539883, -0.0989156640829265, 0.3058095872914767, -0.07690327023830718, 0.21952232534900631, -0.7239779739846423, -0.14951178549083013, -0.04013747409037643, 0.4114659047882258, 0.18093158251811203, 0.08212128542926261]], "dtype": "float64"}, {"__nd__": [[0.43132954615435426, 0.06528278532980084], [1.263374754426895, -0.3693128686391923], [0.047564642712914135, 0.7398733557320017], [0.3136203829824532, 0.12557251467786829], [0.4067615964386362, -0.18702149228646542], [0.04843334756084192, 0.07298523553840043], [0.7842787650772136, 0.13489839077769067], [0.8397719401099629, -0.006992036955087221], [0.11093244759837236, -0.6819925259469188], [0.256686202119088, -0.7931806532908846], [0.2746234757408241, 0.3988670325397091], [0.16575517396160397, -0.4196367100947684]], "dtype": "float64"}, {"__nd__": [[-0.11215572033733304, -1.0668495232683268], [-0.2889561133083265, 0.0950209546827912], [0.0012413648480324932, -0.14987798767489957], [0.5247327349793947, 0.1384226959414086], [-0.0038059286341878976, 0.07270990450713362], [-0.4334598088994846, -0.07742682356730003], [-0.7955603115824792, -1.1194859621734543], [-0.15091625391513339, 0.36251186007824476], [-0.4546707390445569, 0.14519272352863874], [-0.7159033834547669, -0.022654535584202953], [-0.004711240859924028, -0.014472417179607278], [-0.5928665742712399, 0.21052356821059115]], "dtype": "float64"}, {"__nd__": [[0.2688961622724186, 0.20762575599172553, 0.9065692627940165, -0.15108248008308608, 0.06972302041014942, 0.4831960331899793, -0.5461797188180828, -0.3734856343690185, 0.4375163244354939, -0.1436660710488902, -0.32682663590543237, 0.19440464458287723], [-0.5496171662203947, 0.05526514285552478, -0.4202640776830031, -0.26490273278444576, -1.0722446284757856, 0.08277271048269541, 0.09672159623231723, -0.07430912727609827, -0.27030207116731714, 0.09133798094995826, 0.5916236083072282, 0.35173411437541413], [-0.28663609889133873, 0.09778488235099664, -0.10080185960150988, 0.21818456394823071, -0.3215304400944432, 0.600920605666758, -0.3121167369987907, -0.39929279479446067, 0.14319087636464234, 0.6644560962120301, -0.6717022606865557, 0.029276721962965616], [0.25890028365215223, 1.0910717757875132, -0.5975307856042729, 0.4603892116516421, -0.2870375964308943, -0.12559675913391452, 0.3912290344355174, 0.02925881848831101, -0.5405236479185266, -0.08121800025429911, -0.14918604835356736, 0.07785368146416523], [-0.37351017805385733, -0.27488876714601246, -0.133319878190806, 0.011566940728764546, -0.06738154676703706, 0.17330980335971313, 0.25731114929018334, 0.13096892274676605, -0.26628106486082, -0.6253089864753136, -0.3604863529972797, -0.27960645672677353], [-0.11213841962837912, 0.3565057816917581, -0.04465018843362615, -0.3989569492302263, -0.5466016396247578, -0.06840503779842617, -0.12492403912873058, -0.45099726794386846, -0.6844563878504081, -0.16527852274004154, 0.12826668131172, 0.0555785821282857], [-0.6117033594330991, -0.2046482671783414, -0.10788698678538189, -0.4181775149053302, -0.06552245439633213, -0.2880785977151151, 0.3390569052022013, -0.29267835486436966, 0.6193839444252063, 0.3782969808063163, 0.13068407243723573, 0.08525171729132174], [0.5483597389997291, -0.15660042582932088, 0.15893655171199383, -0.1064112189994368, 0.6086515945129704, -0.12550659287707022, -0.10584430430512103, -1.061802874377659, 0.09314876269438166, -0.14992897184904713, 0.2210540931033893, -0.593594935902149], [-0.3550613205559342, 0.517108059155901, -0.31399439587418554, 0.07232189566391616, 0.3597727669879081, 0.025609440711857084, 0.0185967352579916, 0.4790330236918874, -0.21833504582320842, 0.22520167606350527, -0.11206447609035358, -0.4555078434889425], [0.01469361568593678, 0.5293233215339813, -0.07054602911311374, -0.28118245825570864, -0.08442632486508857, -0.19563186138986166, 0.2027447858571092, -0.4031086629520275, 0.35912475111461273, 0.16452784953307667, 0.48971068504233317, -0.18399662387472424], [0.3765756482708138, -0.3996006663830111, 0.24146698416952816, -0.09046023114424903, -0.44834776241714414, 0.31646843079270864, 0.582559912877455, 0.7165381754641695, 0.1821040234427449, -0.09515702662282394, 0.5926082370669982, -0.37923091815618754], [0.057283596995108765, -0.5921965702288177, 0.05786889888785212, 0.1035709877010041, -0.16221723065758492, 0.4199440019331365, -0.050714392312921866, 0.19089784400914794, 0.2163539099903185, -0.04882303428455928, -1.0265170979684552, -0.16599410090136846]], "dtype": "float64"}, 6, 1, {"__nd__": [6], "dtype": "int64"}, 3], "e": {"__nd__": [[[-0.07660242802708098, -1.3208031396630269, 0.8744954629365638, 0.5591499546351962, 1.6709239687716828, -0.1857964443375822, -0.4752727759749124, 1.7939703123323965, 0.7203192783939583, 0.13499809934745724, -0.1508115372392978, 0.5968648397346621], [-0.1636129619483486, -0.3344040173511303, 0.42699367303260055, 0.7428518496725534, 0.8619292850473017, 0.22504537754347992, -0.3134640492904511, 1.3479178361372068, 0.5851502112635407, 0.4809798459134707, -0.3980384547096059, 0.5995214710000477], [-0.26624385311634596, -1.4417687427937305, 0.8219140813464347, 0.3331796450889616, 1.8953250972080011, -0.1073000979010724, -0.23063804182581626, 1.6090807956044673, 1.08354360388047, 0.4172540279992824, -0.13993473000871137, -0.014803346823347996], [-0.03595461112623857, -2.83434043771574, 1.4975164860341093, -0.04724081755834597, 0.039605260109494056, 2.123604207155542, 0.8450326851200383, 3.0950974198555405, 1.9020146893800338, -0.01847135335511376, -2.1023738261781513, -1.6777328848808302], [-0.7385310078082997, -0.671238230094569, 0.9876704720944948, -0.395379244164037, 0.17606101181344191, 1.7230273650206611, 0.33877115225790627, -0.7051344558325768, 2.4779345748755626, 1.214336110907175, -0.6698148868431408, -1.3266458388091915], [-3.232224180145267, 1.5114299304412853, -2.7376155374156133, -1.038830526794805, -1.385575968334459, -0.1350971626042084, -0.05048816175210463, -1.7404122314534858, -0.032874607129904726, 2.5992444972921485, -0.8809472476796784, -0.7709230409338603]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[0.16074282234753837, -1.020789733678078, 0.0474276908896784, -1.5008867622009963, 0.6206301233186688, 0.6164167254131651, 0.39739220078567467, -0.4934579844854673], [0.2822477794349774, -1.273610675763771, -0.7967376177796717, 0.4111950662072986, -0.6933546971771953, -0.26668671647424125, -0.039175923030463966, 1.0462981260717192], [-0.6963016995762141, -0.5843914845879179, -0.726826953459274, -1.683040493249911, 0.16197092821697653, -2.103029375240034, -0.8465458486461215, 1.265973282693592], [-0.026277842447757927, -2.3051788533615616, -1.0430693525079087, 1.6285830421248864, -0.32531706249155684, 1.8127764286562436, -0.6013218239671063, -2.491576416886229]], [[0.21890463990196374, -0.2836671346201859, 0.23245204517865223, -1.1036115204181893, -1.1257826887904667, -0.6452894692848569, -1.1196916909498043, 0.8037200931092185], [1.0661982327744342, -0.2702969200071156, 0.8916669835695286, -0.7507021022932003, -0.36554094076831367, -1.7271910960827968, 1.2042310479088014, -0.5892519712947163], [1.210961682920171, 1.2067854053441773, 0.48421702332096667, 0.1988880642143254, -1.0628063082651829, 0.8389934924461099, 0.5972555401279144, 1.092382312749999], [0.33935847298117433, -1.2334654140941401, -1.141288332499505, 0.934946380924128, 1.642264164564655, -0.2568472268344556, -0.0704347369084335, 0.3224594991766396]], [[0.2091006403744334, 0.29826035209371626, -0.2612937119119749, 0.1878905580534524, -0.5783578064067194, -0.12147719772239406, 0.6596133416059291, 0.038244239593292005], [-0.6475821885828806, 0.2730965061111936, -1.1245279127075682, -0.6790492974816684, -1.0731337070807703, 0.2575822381158148, 0.6163615359457827, -0.39491161149627063], [-0.92282761080968, 0.44186762946287383, 0.22130244716284322, -1.602532607620433, 0.9381987461961482, 0.8914095947971984, -0.29408915051741596, -0.10889468959998472], [-0.14413360973190809, -0.5229936598085163, 0.38589642217543385, -0.775660316318777, -0.35924982713267045, 0.08548369872045657, -1.7490529331535676, -0.5796825133917982]]], "dtype": "float64"}, {"__nd__": [[-0.5288907771204006, -0.21515313180950146, -0.23841312537653364, -0.11577219417553111, 0.014918266463352346, -0.4533915644463011, 0.06941377799014507, -0.7343362544333029], [0.32171727023366087, -0.08489126421720715, 0.2610714405422684, -0.8199420940748106, 0.49414745924806686, -0.17432040796171952, 0.07592697212580957, -0.013842301058011087], [-0.08487519035174865, -0.24366833334813293, -0.857675512344456, 0.4424333600889956, 0.6928308526085336, -0.2943485604556528, 0.4883653316226241, 0.3303212360538805], [0.09753300106709253, -0.09546578623887537, 0.1985839313757083, -0.2965460103686716, -0.17030183022075676, -0.24935191360677453, -0.3330931261259546, -0.25440915560026306], [0.20732820424603499, 0.23608030538392227, -0.35520364866262316, -0.05995048639119906, -0.10822447255641428, -0.7264640479318086, 0.41947467078931056, -0.4674362710825699], [0.42728019521421196, -0.958880157935772, -0.33697062879866874, 0.28187165126242036, 0.15331903746143893, 0.1809683013606885, -0.03722395222127409, 0.8046882530854476], [0.008786796890639166, 0.11890344378401654, -0.30607560590431904, -0.33442848140990905, 0.5369142095561222, 0.5399193999989861, -0.7362261809433475, -0.2329339895348103], [0.4932910210171718, 0.02802656555725366, 0.3848303517913606, -0.730114687801295, 0.06991796876043842, 0.002973531219994342, 0.023918646047764815, -0.2630938777573284]], "dtype": "float64"}, {"__nd__": [[-0.016986879086425873, 0.7128934254957812, -0.4338550058275497, -0.027789114419538276, -0.12371488448603202, -0.10764336506159163, 0.7277867418065113, 0.6383523563094714], [-0.14729210690646619, 0.21543206800974227, -0.32822144761258715, 0.13169572848800684, -0.14803985400972666, 0.5983872993861326, 0.1862042547221352, -0.42257935205607694], [-0.7217323317568961, -0.4490468375160716, 0.8561518326279002, -0.39075692924172273, -0.5704392705708565, -0.27112814395113594, -0.1959356012191617, 0.252067050949207], [-0.3039872175247864, 0.6767260607764811, 0.007153834495963778, 0.12669827100844372, 0.579943357071417, 0.07755599094544034, 0.13561683878446593, 0.2649888073502687], [-0.28900916378991226, -0.21142442730006566, 0.2803167665728367, -0.3103371250183789, -0.22596537402933575, 0.38314895483883066, 0.246322486215038, 0.14344775204270852], [-0.4546301656339047, 0.8031135936593223, -0.055954503988465576, -0.12326071372082106, -0.0681981543907708, -0.33558410440998454, -0.10661557403314254, 0.8108557081385485], [-0.606479695112405, 0.9054782348665422, -0.38354513585149363, -0.38379988201939425, 0.550526097029126, 0.7723134800620524, -0.16284554020581776, 0.33197007289767116], [-0.3188633711574649, -0.15120056597583376, -0.3128398692298881, 0.042140309532004094, -0.275043669699593, 0.03819902923259087, -0.15694242045055803, -0.019804816760711733]], "dtype": "float64"}, {"__nd__": [[0.005958437052602141, -0.29630748189326833, -0.18150227319080592, -0.49259612531890823, 0.17336539779550772, 0.2669919086108175, -0.1802983227039287, -0.14973846662562582], [-0.25766815174627844, -0.22095100621652872, 0.020541300032351894, -0.7188929843763112, -0.6254827470800107, -0.01389733830160358, 0.313994036670356, -0.32226589383109894], [0.2775582603179655, 0.33656808784706205, 0.0002448127686131359, 0.851197316648252, -0.09085990803247973, 0.010567289054704549, -0.4989413680570173, -0.4554927804939721], [-0.002083171685759258, 0.18743588617738824, -0.1302812755306106, -0.06916879342575226, -0.006660252107614626, 0.25228631317621847, -0.7336932589483199, 0.26356339824401137], [-0.3933277944193614, 0.13890886448225168, 0.25587692056299666, -0.6225796186309038, 0.3845831759099639, -0.03960565447635814, -0.2818957402731292, 0.6469626967514395], [-0.5780276392263554, -0.6768644165692528, 0.0029288315544711498, 0.10021812223578593, -0.16266728074717463, 0.15560614286875066, -1.020085829066864, 0.32549734655803286], [-0.0022386051738431663, -0.5046577598910954, -0.5511710320855076, 0.11788848382309934, 0.09276402473654632, -0.32817450413774657, -0.01873502695333764, -0.0467360551352313], [0.37344495605990086, -0.4292305453789876, 0.3521799557171193, -0.009298630471773905, -0.02853819160486335, 0.16723451431652459, -0.2384316000313004, -0.6160165765064843]], "dtype": "float64"}, {"__nd__": [[-0.6990697446826861, 0.3651347854879091, -0.7385804870703958, 0.37526193037731226, -0.3250641254779664, 0.40070012199285054, -0.4097640482862597, 0.08397372936279739], [0.03653926008370205, 0.27621339282237434, -0.19415997554596695, 0.14579033273445793, 0.5988297737020815, -0.5779939890866944, 0.38311500395221737, -0.45397261444741455], [0.3618922744272761, -0.17939819146353309, 0.5536922447291214, -0.12615047489998737, -0.22448959425869697, 0.3964868336557962, -0.11849267309781551, -0.10080152864373673], [0.3651951723094477, 0.5570961919931942, 0.3320624893912565, -0.7736134149052518, 0.1270872799360063, 0.1116543848892721, 0.3437187580033304, 0.4225292676352761], [0.4121973055187984, 0.20105834308079046, 0.8211929396298979, -0.046999029285441946, -0.25166838799714997, -0.4863730036273823, -0.1328322055056897, 0.3344891954325382], [0.036090596931815225, 0.5884235083978434, 0.02536789420079933, -0.1853458621157819, 0.63947156362407, -0.047787654760667195, 0.4914775700351095, -0.8627303426240301], [-0.4887207855676877, -0.17511449844396212, -0.38123027893451833, 0.13286896919325825, 0.06596166946343433, 0.12364203300800608, -0.31994845293933877, -0.31243856860475183], [-0.22903333043767576, 0.6255087045591315, -0.08824254937479108, 0.389954131622459, -0.3073247389521307, -0.5502025448475982, 0.06465445700746812, 0.3959782783686781]], "dtype": "float64"}, 2, 2, {"__nd__": [1, 4, 2], "dtype": "int64"}], "e": {"__nd__": [[[0.6428770925088703, 0.39130387023686736, 1.1928477895198242, -0.2925275894392295, -0.7419219963336975, -0.7619957430301245, -0.06069039892050983, 1.356775455608498], [0.0, 0.0, 0.0, -0.0, -0.0, -0.0, -0.0, 0.0], [0.0, 0.0, 0.0, -0.0, -0.0, -0.0, -0.0, 0.0], [0.0, 0.0, 0.0, -0.0, -0.0, -0.0, -0.0, 0.0]], [[-0.6658575056054594, -0.311499658036362, -0.8945655691799286, -0.753109275178368, 0.5941490595442662, 1.8961979062054422, -0.5621314570751692, -1.1612247322956977], [-1.3130103200914092, -0.08848015900878348, -1.6781980868672992, -0.18864179875170406, 0.6547279109849201, 0.7812875462017835, -0.6375264961504775, -0.6473478020624321], [-0.6307216274033032, -0.15960877838608284, -0.9988989324313324, -0.5446026667904761, 0.44090481812265786, 1.348524815599888, -0.4269694169661138, -0.5668534238904276], [-0.8038824932596609, -1.3330307435172681, -0.902555761915417, -0.232404667355339, 0.32996089188573857, 1.5407154953402218, -0.835942784675265, -1.0925807757410868]], [[-0.550929679754344, -0.49920951818532305, -0.67348840853519, 0.060706655864049164, -0.1106903562420313, 0.5463772379568455, -0.39282478564316176, -0.007589652782769088], [-0.7595714720043305, -0.7135650059702631, -0.816100231668011, 0.11137911568947258, -0.24912412239225645, 0.6030049211824307, -0.5903838816897052, 0.10674587438662325], [-0.0, -0.0, -0.0, -0.0, -0.0, 0.0, -0.0, 0.0], [-0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[-1.6493218318181517, -0.526802098360025, 0.6281986890854655, -0.26869563211452424, -0.30805617768989124, 0.7574686634896459, 0.14817861231024435, 0.16542200407237162, -0.5176767079238197, 0.2570085848815919, -0.5238391438447193, 1.3991074013982, -3.1044430386921227, 0.0945834605675184, 0.7116065861105035, -0.9667118316399859], [0.5773001505101631, -0.7307002296676894, 0.6795280137945875, 1.6355239416121032, -0.2545260222859596, 2.205621795554146, -0.042973651593618364, -2.4412008841221535, -0.4305480403495739, 0.10364095159196587, 0.19742543064492785, -0.04644368564484061, 0.35097651528548374, -0.2243554993708294, 1.2972178930577696, 0.2887120501523157], [-0.17465293222996037, 1.7975663402840383, -1.3163118707832868, -1.9239384241862865, -0.7279305755762397, -2.5956839160717284, 0.4604748573193946, -0.3473090762218554, 2.063268457104936, -0.9786630248183872, -1.2366729373677896, 0.635145280001847, 0.5907925095619063, 0.6226951058913676, -0.6258737079878895, -0.6309187270648428], [0.502020179095927, -0.20058742897900006, 0.5315168257370777, 1.0815270072626375, -0.3271239266895366, 0.5293330307405559, -1.96260609636284, 1.5557239773506355, 1.4922387879812353, 0.1474225651032618, -0.23429111338332909, 0.14019831552598402, 0.25831418472209383, -0.05360828573334312, -0.6254017779960194, 0.3155295482740563], [1.2422159627084581, -1.886293675667928, -0.47751280060377244, -0.7891485221121012, -1.1116383330201092, -0.0010254876003088814, -2.1782668480829224, -0.40858407975287214, 0.856632154440427, 0.9037534153992262, 0.6848056303941823, 0.34760713294877216, 0.8362714917149559, 0.17232786476123388, 0.5112304994068312, 0.7581361743280601], [1.52588914369472, 1.0757631412877162, 0.9365333427672715, -1.4396315838120477, 0.32189321097933693, 0.21753999153080233, -0.4869099359918372, -0.9868846099516582, 0.6954519704304335, 0.12179696034040671, -0.265096608571025, -1.0615436393643174, -0.7206279086590425, -0.4973069762069178, 0.32830760276717613, 1.6568144692019835], [0.2117931803349737, -1.979476762616457, 0.4335184394872259, 0.46364250836208476, -1.0189481378840013, -1.3299508294844882, -0.003824250814567051, 0.7289733430302223, 0.3903792478973071, 1.5290593975253812, -0.16642716642792402, -0.026151819974327598, -0.9110639204622357, 0.9278238213461044, 0.15830590241509288, 0.5025833378899144], [-2.3212164908070045, -1.8414589990282537, 0.2979131406372695, 0.5362416325774639, -0.4973833834888324, -0.3860222930946968, 0.6411127508051163, -0.8979023510047629, -0.0714998388805662, -1.0428834579704784, -0.680851928661015, 0.7307190295145215, -1.833259673344813, -0.8398033739458514, -2.581917856436749, 0.2262376767228562], [0.10617378070040351, 2.7896872778701916, -0.4856132335279592, 0.14001342537380168, 1.0045424773776788, 0.5970952576072437, -0.36468839994372015, 1.1571643665489335, -2.8006657020610444, 0.4018033739491583, -0.8237016916476322, -0.860269623280194, -1.1325375310609747, 0.24729779524653508, -1.123459367643485, 0.05651009629857553]], [[1.6345884448899073, -0.8203730687875306, -0.0005702565214660535, -0.7058470885799282, -1.4266785058524654, 0.4707367931990331, 1.1149262579253518, -0.7815501705813123, -0.06131651980059948, -0.256969704615265, -0.858002409997825, 0.7094393281485896, -2.123163417056533, 1.3732467562995212, -0.6638829833796918, 1.6404232239724166], [0.376624682434677, -0.20575820147586033, -1.8112331848934737, -1.4524771346589915, 0.5415994785444427, -0.8737642587872956, 0.750043443540419, -1.4272911870671228, 0.932664833814936, -0.16561097688293597, 0.2519615339633171, -1.0359212698998492, -0.579721781939375, 0.35120880415860795, 0.004969663124250811, -1.123652994104842], [0.04656178759285211, -1.645364497841874, 1.8568570357849234, 0.4020678155934242, -0.6669210856534941, -0.47116709776357313, -1.1940113181616072, -0.0838747834126649, -2.92103783256006, -0.19710490493761623, 1.5188907677438608, -2.2817659143549376, -0.009886240509446068, -0.732597854754713, -0.34930012643191505, -1.3488909640948055], [0.8098892485578755, -1.467680266142356, 0.8046097799405129, 0.9090008127771954, 0.05650490678347922, 0.4366960853741022, -2.1150569813659645, -0.8103950601547404, -2.6517663781983587, 0.4797347802758453, 1.0098332097989668, -1.6515240032305838, 0.06291495681354617, -0.3780658193792855, -2.0521302265370736, -0.4941888774835283], [-0.05300423372450134, 0.8684529904829009, -0.23716526613132108, 0.5317512685170399, 1.8294305308620598, 1.0448269752144979, -0.8796429001884848, 0.45014162807110025, 1.633497514453226, 2.698733006921393, 0.7433814409913445, 1.1572036996594162, 0.8913856742899029, 2.063788583912317, 0.9464554517806758, 1.0975731078204325], [-0.9135309785787551, -0.328651489063961, -1.543105881059299, -0.8915952041865093, 0.3194286431626511, 1.1315746213302662, 1.0837029079463592, 0.49628248545124914, 2.20537730971299, -1.2620900000797421, -0.001466859401565408, 1.5251878617814563, 0.4125178251010986, 0.20514497668094936, 1.0189681688438468, -0.6945900254591005], [-0.6745295301651842, -0.3435284765839912, 1.9085383093549495, -1.2610087139053043, -0.5805574885950892, 2.3309009822633033, 0.43690458099310636, -0.23002944435833342, -1.8490454059288792, 0.194271393383485, -0.007064847114228712, 0.9929315238957606, -0.8283034096405134, -0.6644617099443298, 0.8811375802103616, -0.9570307974411767], [1.632862759068735, -0.15655626954336238, 0.44791224371866384, -0.5733191002077422, 1.7335369470595274, -0.5630571079229515, 1.3651878066444305, -0.49156538209152706, 0.43324458933831794, -0.8431683186913147, 0.8134686816556859, 1.6235227456399406, -0.48333062536394855, 1.8981027190162574, 0.14106746948648735, -0.7040124104194552], [0.31492454867444963, -0.40900293928572723, 0.28183077833071724, -0.45954332353001137, -1.5527437282485679, 1.3016634727104162, 0.28007877636012907, -0.30848157925269004, 0.6530278133913737, -1.34920730543474, -0.20185913758904445, 0.7462833155559788, 1.439537701151859, 0.23093424412287208, 0.9131057111446131, -1.7256338884547155]]], "dtype": "float64"}, {"__nd__": [[0.008880572711461822, 0.3672012866137278, 0.5541884327807108, -0.4601401459568997, 0.6597050202371945, 1.0216539783195517, 0.10031950075518066, -0.6742324504313157, 0.2938207184172236, 0.28550091179977816, -0.15990669377370625, 0.5792011778875146, 0.7447545473625677, -0.39243043546684153, -0.4118977833362711, 0.11474992543171213], [0.4949126188773092, -0.5027652483807592, -0.3188825684460148, -0.11642096189559369, -0.3410227505909165, -0.10883348440370633, 0.39036366535481615, -0.37406710736337656, 0.27951936273021166, -0.013660021291983546, -0.125092123466834, -0.5809917267372803, 0.08032486489791059, 0.20351599676839338, -0.339962679707264, 0.35988747529566856], [0.3961268028295821, 0.05423097407523428, -0.366239345635864, -0.05004681998616465, 0.37627048458974727, 0.3663894796494097, -0.037882296490852535, -0.08954010706299781, 0.3035734688180189, 0.2723318455889488, 0.4170985745104749, -0.44131565539556655, -0.0448568024962766, 0.3659127667817259, -0.29038113727786785, 0.056099034147375784], [-0.2894445571076451, 0.40059185026488814, 0.5461914154761202, -0.2007236135306188, 0.9723357908297274, 0.3538863087157518, -0.10730150337439096, 0.23064959357523432, -0.16674179849687149, 0.4408598641784761, -0.040316433373203925, -0.3303421478665678, -0.08277344005361818, -0.492582560298791, -0.9093819594136658, -0.5630003545429548], [0.22711306611470758, -0.2248730894591558, 0.1568826689877837, -0.19063869277845902, -0.02313480758512278, 0.12087398459853259, 0.34388314497249695, -0.39610900263977966, -0.34440774325296186, 0.021210062575347403, 0.6394123712732478, 0.11214290058267372, -0.07399498452466563, -0.06343336345295278, -0.097682859542565, -0.24720975486816932], [0.3586957343921315, 0.7603075918448807, 0.11565779999041377, -0.5776665695514637, -0.7288552143967421, 0.7901175576623979, -0.6968443405610187, 0.9570196433363414, 0.19833940767572955, -0.04796894258842291, 0.3506321141796558, 0.019842740063652425, 0.6820629838015981, -0.1884989985783012, 0.13519409764647608, -0.2073424794578711], [0.21923454343378426, 0.3373833869356496, -0.5206901353203583, -0.6512158859292018, 0.4111396334737303, 0.3236968091499485, 0.19741122814751605, -0.2790589866087614, -0.461816709875626, -0.40385514497289904, 0.013077399106088496, -0.39068050093941187, -0.39415305391059996, 0.1555844350427279, 0.6869939343646017, 0.7917809770450792], [0.11242431713913865, 0.4781812274742519, 0.2117062769088913, 0.16625880519028724, -0.017340349986112287, -0.23386477009620033, -0.1974738317770739, 0.1141940059798708, 0.604918528967956, -0.5280844730757814, -0.2099904222972247, -0.45329426205886625, 0.2879931193426975, 1.009510756622699, 0.211316582174916, -0.5148758511726389], [0.2078255433386978, 0.8209643504054152, -0.3075260377719385, -0.9168234165761512, -0.8051476514880289, 0.19197680842515832, -0.7711836846408033, 0.6306875769247675, -0.9763551067769648, -0.06751283949835718, -0.5313363294654362, 0.34754042226620663, 0.14507060298659605, 0.2531760269612873, 0.29612937140548107, -0.33176912172561623], [0.03748233592607993, -0.37720637159932613, -0.1546858324584825, 0.07032759092364438, -0.17275673824831939, -0.5785427534980594, -0.5475059855132467, -0.4869182690523376, -0.45873612019066273, -0.4812911263812664, 0.5506195605905381, 0.33079097674288216, 0.11536116748622516, 0.5605447758626092, 0.20504119756267586, -0.20421723447510376], [0.4644394320623623, 0.20183835263789973, -0.861058790618325, 0.9161929718134243, -0.05855487203646573, -0.7181496641936129, 0.5347461255887057, 0.04575970049321945, 0.05775573328340047, 0.5791146126151361, 0.3511161010577577, 0.13297586318788465, 0.09017201268733162, 0.039686130043420545, -0.07326742586935918, -0.563326072535986], [0.43877765949155373, 0.0002086284163600807, -0.04669515259118544, 0.3060254337315511, -0.28281109162743195, -0.17070794831983216, 0.27507581006872983, -0.09454275708511135, -0.4529626886178732, -0.4341508471570998, 1.1872060667126645, -0.230382729298249, 0.9026596856304553, -0.3032915848966149, 0.14393379221481256, -0.19324912027353133], [-0.4880457498041598, -0.141875386582113, 0.8455659255266449, -0.3260464346922544, -0.4281840881508274, 0.1513477372600257, -0.42829565502196243, -0.3480403974203316, -0.6088832841650731, 0.38287324672047063, 0.2635812768034748, -0.9961561181660139, 0.2615986854178459, -0.11728455786941488, -0.05402903577285674, -0.3260186900416791], [0.06922866926745036, 0.23710798402461683, -0.3530317349685093, -0.027288736434723977, -0.04285106885193506, -0.35373387758670827, -0.24655806677190506, -0.06884519312848995, 0.3179547352877493, -0.2962043766226768, -0.017504456431152872, 0.3903552941777106, 0.5446101732894947, -0.6114903260636577, 0.06753130687551606, -0.11565345720309106], [-0.24142725007229476, -0.3120994599150054, 0.670975228193585, 0.39180424762481597, -0.1233162942710508, 0.02535671609324007, -0.047939352364184035, -0.09538399094986973, 0.31854168120257315, -0.20422560901974354, 0.28208003870288534, -0.5923797965086087, 0.09305389394388135, -0.05682212152257487, -0.24058206046524064, 0.1638790730161931], [0.048941602232563136, -0.13580517578697135, 0.5354010772257162, -0.05313589643432109, -0.078960974480408, -0.21901614669386174, 0.443595356602995, 0.38352023545870084, -0.951778803060816, -0.3692297871149662, 0.14335868385384956, 0.3854362949740174, 0.20809408439983376, -0.03699040116623309, 0.5959275236865189, -0.20153297066116274]], "dtype": "float64"}, {"__nd__": [[0.04951804103645919, 0.012460995798072727, 0.062263026010589244, -0.3649981757847711, -0.0962997622726458, 0.296402878337396, 0.42115160949345287, 0.34672908421046367], [0.20838350763556496, -0.09101739589248203, 0.2774899113096274, 0.0039093639806586, 0.07368230739515982, 0.19761325957887366, -0.3126614649386139, 0.24565933886109137], [0.46766881751632144, -0.6303004807357807, -0.11089021378333236, -0.13180339888438627, 0.6543779296178955, 0.05781808138762273, 0.2583136989217933, 0.3109173769558407], [-0.30073098683398825, -0.5186142225086359, 0.4628915460469603, 0.526907684276236, 0.3629311591107034, 0.34017160045255745, -0.2759535472434755, -0.2804095260920354], [0.012192044441296353, 0.27359590904846126, -0.12316883701569888, -0.11366805445054362, 0.2037431168469077, 0.6479885414029928, -0.6247357491473579, -0.08392569723456396], [-0.5925663373081168, -0.7198946382651106, 0.4561260247184743, -0.4031738899600897, 0.15103887062782942, 0.45492692128309964, 0.2838452196538958, -0.391465884840744], [0.15524663637092567, -0.025294979299465267, -0.27989062046136015, -0.1263762469857021, -0.21078631464853906, 0.4108880526611263, -0.004544691671654007, -0.1629693736661672], [0.16370616465190554, 0.16389136348229813, 0.24424811701688973, 0.40268839723414795, -0.29882234712041006, -0.2169802304211047, 0.454778860329051, -0.27425227909988664], [0.2856404011787237, -0.8834613062914936, -0.0077776570750057675, -0.34371044105203463, -0.08485813489565092, 0.1618704537790875, -0.7909335993048319, -0.6143872165184525], [-0.015393390259665735, -0.47632940573730753, 0.0870919711920213, 0.024735300186089045, -0.4233375665441747, 0.3650485880525231, 0.282749155899322, 0.16916983945179473], [-0.23451722824310958, -0.22353666743727874, -0.5720215762975324, -0.8330566994130696, -0.15517655997660557, 0.2823282443159378, -0.10854488948012948, 0.2269636465834468], [-0.23120709588643773, -0.18931004656963313, -0.17968536115563977, 0.47999626787188626, -0.4660175128054516, -0.28213871445611194, 0.2990469402894592, 0.38700411716092287], [0.24234352716148982, 1.0474990859415045, -0.051343528644263094, 0.005978962064412528, -0.22918612344479491, -0.1266526971594695, 0.7070639717422467, 0.4258653894428693], [-0.1314481128556943, 0.5318390919992896, -0.3686931384525159, 0.7387913992540008, -0.1475367963950674, 0.331860478831165, -0.5192009030981427, 0.07347719776213801], [-0.15920341376440755, -0.08633106060967366, -0.0374674274159142, -0.6706290352964032, -0.6570314559439794, 0.5581608784973721, -0.4660568688161446, -0.8918180181916665], [0.18027758545598704, 0.0030910517945840414, 0.06718497991765991, -0.1794831204013193, 0.2743683633372717, 0.36984804965951495, -0.0848944596672152, 0.16646360745746244]], "dtype": "float64"}, {"__nd__": [[-0.7783405419682898, -0.10385045258657617, -0.36282664331494097, -0.45783534408961135, 0.38294131680510224, -0.8613165913731674, 0.09620231926173177, -0.5566218388679904], [0.44158315040603713, 0.05241775059704873, 0.3611194931979374, -0.10820203719144855, -0.5823866120377393, 0.967307229293264, -0.06987775559889443, -0.4545952382152845], [0.4420602005451266, 0.8465730095178767, 0.012421928269942274, -0.18587692759577815, 0.2722237087199141, 0.3633812947692244, 0.3961635153474925, -0.11004352957691677], [0.005807972054077318, 0.19809878911683115, 0.8172292642148222, 0.5328291687606307, -0.21619492208287336, 0.19367594339747726, 0.3598792367285, -0.22248379110832453], [-1.1593950876890433, 0.06443582770108695, -0.1986266093153433, 0.11832915645328515, 0.17151728661632568, 0.07985328286728234, 0.03207758067854586, 0.4628771472536062], [-0.15441313695285722, -0.5796557533707271, -0.8943873445976112, 0.24452540241950788, -0.520760219542848, -0.037870586743728056, 0.053453428059937885, 0.35599536961720535], [0.3850920392754989, 0.05602641911824267, 0.05704105017445779, 0.7074938617074289, -0.8265697203102322, -0.05439213803474843, -0.29333953199005863, -0.02662335843751126], [0.7553110076648729, -0.16598373955316145, -0.46829312861868955, -0.5519985226897202, 0.02335605252886063, -0.2521692712411287, -0.38722730408941836, 0.5068972757204946], [0.8849444932025167, -0.16834910531288833, -0.3215094216165386, 0.28521839299284446, -0.11276700022489046, 0.37648928216091543, 0.4876657548744642, 0.23394177603433364], [0.10243410134829123, -0.3193494381298166, 0.4405046440745586, 0.5320888695868607, -0.5157838769470461, -0.18879817155703327, 0.2589550736895003, 0.15641481338645752], [0.5306989192995034, -0.3524449118366184, -0.048539620456497484, 0.03884586549188466, 0.05096160554346851, 0.4941432867031299, 0.3798347784267841, 0.646954376195597], [0.030861625476137328, -0.15883361003319707, 0.39000130995304355, 0.22166320930832628, -0.4044954374846901, -0.42338537849875374, 0.13268097314946078, 0.4429892749036811], [-0.3552042287273933, 0.7649406071012579, 0.28375774621591426, 0.054202082332405026, 0.5262259411549172, 0.07541307349557819, -0.5405765713646944, -0.15198451931809587], [0.38401494806817565, 0.3942161330944991, -0.15048640571066743, -0.030065511466888864, 0.00664057939364295, -0.7524834761967365, -0.2809284922460137, -0.244664268725223], [0.3380021336944771, -0.518685839102825, 0.32280432027267086, -0.07427250450577284, -0.07372364889652301, -0.7555246483605607, 0.6323922343581003, 0.9272970946375273], [-0.6324178937941437, -0.1856852371830475, -0.9254524495310905, -0.000503177804465942, 0.044127910039293776, 0.07302521685503192, -0.5907549891187034, 0.8396840749430018]], "dtype": "float64"}, {"__nd__": [[0.37629878327308663, -0.6826461057384655, -0.32772469831363926, -0.12689895321398034, -0.03694172979901777, 0.6924839063984449, -0.4516618426407164, -0.2332189666819874, -0.12570023019960705, -0.11952762016725822, -0.03802931290997287, 0.13529041882141266, 0.7462578115933826, 0.5731530968628288, 0.17918464000093404, -0.591719852504873], [-0.39054709185449643, -0.38843477000544424, -0.06388451441532077, -0.47440503321120436, -0.26284488089976593, 0.33695664801170816, 0.5251872329818353, -0.07384466987680148, 0.6121216616797827, -1.1604205132444498, -0.008701493719794146, -0.24282940481015614, 0.31081912826516384, -0.05690193941692206, 0.09637590380604172, 0.2089318189643537], [-0.0740644726680141, 0.24750438328903168, -0.1857443518321018, -0.5053840216001669, -0.2706710718760367, 0.07153190236622334, -0.20337952362275724, -0.3028486879637022, -0.7073432699860435, -0.37817915461760604, 0.25232443464298915, 0.15844155070705446, 0.31265300180572264, -0.17286231491200219, -0.09077649639042787, 0.2901018814030219], [-0.45566101617983235, 1.2348297960292633, -0.433174735051961, 0.2309127429140491, 0.6589698379105823, -0.28947508775049235, -1.0492468213417463, -0.6325168715021291, -0.9198449913421736, -0.9511462826796659, 0.754497460119581, 0.37747745784426284, 0.33764013896453143, -0.3938324576966962, 0.01420198150142866, -0.20175336677611552], [-0.12211304803390222, -0.11402862995650105, 0.4368328611385553, 0.13178397329648464, 0.0686573662557723, -0.08729356014698818, -0.6742355273051811, 0.18153768313995308, 0.032279448804079396, 0.7517543000407172, -0.765082541378217, -0.41511645886487225, 0.016595164476092276, 0.17773810442978843, 0.20833730060371575, -0.035473264894212174], [-0.24533212208864033, 0.03870189353354573, 0.7489491225040932, 0.3947908738096486, -0.5154102444908959, -0.2145149489635094, -0.7117279563419339, -0.47878035694559257, 0.3734751306672644, 0.5428576337586272, -0.27541008052646715, 0.2952998602847338, -0.4943053848327574, 0.42158429039524714, -0.15861923225354826, -0.06959581371863979], [-0.32990527937020975, -0.23463541666425827, 0.7802730581804505, 0.1767442038584572, 0.26891190475502896, -0.13502960744710488, -0.12035167974962391, -0.1394199172738761, -0.3719282765804077, 0.29442368353717757, -0.41904652197403386, 0.2821459301275481, 0.6063860992782204, 0.3438931005162729, -0.42956868060442094, 0.6777653846143188], [-0.35013799910500876, -0.09570565356904402, -0.20879906161905193, -0.372315857217842, -0.4262229259948879, -0.18850593821248596, -0.011812860346409278, 0.066527765240974, -0.5932374498825651, 0.6355513421782653, 0.10704658838477257, 0.059714711212414034, 0.05963498235422704, 0.3345398246633602, 0.21362377969185228, 0.12045126836065743], [-0.32646164771931185, 0.29257538312605647, 0.006267619665250132, 0.006417240737947575, -0.12405990430354377, 0.16903000619801925, 0.1954936119177054, 0.1484680178061148, 0.6976731817447055, 0.3679017833043597, 0.05498414308875085, 0.2117373056804418, -0.6603169868101048, 0.370080250067383, -0.17341738652616756, -0.30084364773766065], [0.26197440978020403, -0.4136794935906467, 0.2915627556657873, -0.2602822531771662, -0.5659055231177545, -0.07131631906677323, 0.3478088912045675, -0.2256990343830465, 0.26772715845691936, -0.026306512359061802, 0.018708648707377516, 0.41108053195199706, 0.028437385623789304, 1.1555401045245723, -0.11841849544509653, -0.3542843053699842], [-0.800181238921102, -0.4997330353755727, -0.02379114335447607, -0.7355404567001539, -0.599911790428092, 0.1725000137555846, -0.12346087192686533, -0.39799586490958844, -0.7164476329357887, -0.3462176145196713, -0.10041249066132274, -0.13462398297566783, -0.1089367614320159, -0.14662474475028905, -0.19131946783461298, 0.12194025994025988], [0.3719782572718716, 0.2519220178318367, -0.23561653295809257, 0.3361969889509009, 0.19516824890678974, 0.19362811670171143, 0.2624851398412628, 0.25602373013150265, -0.45193829622843124, -0.006082074275002313, -0.1590771483396842, -0.43071367192271315, 1.196631519449628, -0.12339811622996091, -0.32344891874514503, 0.2110813074479518], [-0.10496041573412025, -0.21246652667357857, 0.10186331968734226, 0.273950745562002, 0.16737229940400528, 0.31912443351575576, -0.3374526718361117, -0.37579993639832066, 0.5707399991111384, -0.43009724032629015, 0.2263561839988391, -0.16715574644907727, -0.5615204310241387, 0.24047562271202938, 0.3276315107979664, -0.943837084488512], [-0.3472659114487838, 0.03332183527676783, -0.8319442093289084, 0.1877264994432159, -0.8946013663960364, 0.5680815800451066, -0.45249033256785043, 0.031276693408661, -0.5924438144157806, 0.5136063430129049, -0.2116217554363322, -0.39338767094341376, 0.031056567872598473, -0.13390847401568215, -1.191567628479305, -0.2665264521235014], [0.06827703636508653, -0.39707024919053374, 0.05995265383346448, -0.1258425401366024, 0.3048079037177709, -0.22845914040835946, 0.6011442891647089, -0.25469970230089634, -0.039177658151656324, -0.19808396348582827, -0.4799227532648094, -0.06371852983260293, -0.27528163531872724, 0.32774395993945044, -0.04157286054922146, 0.36815469592839983], [0.3601942870157028, -0.17396234845704836, -0.06633418468385346, 1.0926078923187927, 0.4058532946586219, 0.12104094896968208, 0.7160847952134084, -0.27093502581417633, -0.6240150639134723, 0.41751811874724487, -0.14226130465968048, -0.6492880515314782, -0.2963802419572404, 0.043292915755568845, -0.41491213629649615, 0.04213869255655159]], "dtype": "float64"}, 8, 4, {"__nd__": [9, 6], "dtype": "int64"}, 4], "e": {"__nd__": [[[4.976704584740594, -4.572756571426457, -0.5170780131032024, 3.725725944692758, 0.4055207110184821, 2.11915360394994, -1.4912981257695728, -1.4764573219713943, -2.9040312661948535, 5.262719007824807, -2.9603408530462163, -1.9345307434294838, 0.296775626489421, 2.680824362496028, -0.7973615111094877, -1.6448747266443824], [0.4660974104550707, -2.504749086062066, 0.9292329844003713, 2.350792059707957, -0.8909988667955988, 0.23032031453536858, 1.1725240736502476, -0.9786048085916783, -0.9156805675193824, 4.340200408198496, -2.852666754682615, -2.425750393488058, -4.24162859156133, 0.9769739022533572, -1.8867334724933895, 0.3445059349893654], [1.3723362659701746, -2.1986313556405115, 3.0453139687970876, -3.0934272260117717, -2.2671977832872328, -0.966965868387335, -1.7942066021318075, 1.1756505741584826, 0.4848650969345461, 4.6513760268315085, -1.3427966529179738, 2.7486991979228446, 2.692940084818185, 4.222966536260829, 2.709141985956737, 0.2535746770285767], [1.3949649103258235, -3.5730513505001014, -0.17197925362349833, 1.0945403221745165, 2.264303836465556, 2.426414890746443, 0.747408748804939, -2.3130265098251477, -1.6441682283163606, -2.406888483562359, -2.205172632071805, -1.6841241634438553, 2.776969327161879, 1.756332432655632, -0.4530414923421561, 0.3347106316795596], [-2.165457486666588, 4.077538461047634, -1.1849670597464557, 0.9918661587825739, 2.230053461751082, -0.7968308254971923, -1.0083311574155742, -1.7297152458350438, -2.88767825057943, -1.5724280174194054, 1.1718281826934225, 0.44451667465955264, -2.3113312335362513, -2.508842410594056, -2.4129734927656052, 1.4137465594188001], [-0.06519879969491471, 0.7774487037567297, -1.5728576425714826, 0.1159138034049501, 0.2263015341580386, 0.850207703998805, 0.7273651123647428, 1.1152126782210094, 1.4186044831740894, -0.07005760091284335, -0.39711910445111775, -2.1881472193859257, -2.567834898692854, -1.8657322009183037, 0.11673651293142977, -1.4855007456548965], [0.3347361968923106, 0.9687621582764354, -4.13475392128629, 0.826787089938287, 1.0377822527069627, 1.854064154038704, 4.097241422567652, 1.083238187084277, 0.2973251264455991, -1.290158688425802, 2.0766883398951625, -1.6056173864577958, -3.0273494391560556, -3.6540470498960134, -1.534141802978088, -0.7808891152891637], [-1.4000259712621193, 0.6726341192966608, -3.1426819705558686, 0.36842143798247207, 1.368542929000137, 0.6275582987308735, 4.229419051873773, 0.9707734574154627, -1.70997723052099, 0.6508076363925205, 1.8365076510432432, -1.1058710218927312, -2.7280187190464322, -2.886585193839485, -0.769199336562282, 1.1305779252090697], [3.6997867839632708, 0.7791518214291339, -1.7267601583616976, -0.9923772357252441, 0.24720608469464106, -0.9264619344222279, 4.260882618399524, 1.9318816695913943, 3.6004410349990645, -3.188765876507385, 2.503602374887629, 0.5069416032316343, -0.5383143783161711, -0.35626656402562973, 1.9476729783031874, -1.689376900962778]], [[1.7202802899133816, 2.0545953234174634, -2.834216556956384, -0.8377238286459409, 0.7750198511445717, -1.4059041550722249, 1.1259746819482404, 2.233007739564438, 3.3418773673613047, -0.6356625886604492, 4.152867143219858, 1.1669909434981836, -6.042565828386642, -6.068731666374427, 3.712828168335386, -0.49050757637312326], [0.4427485678943133, 0.12401855428279403, 1.5461091235035407, -0.19806741704339378, 0.6146121411602696, -1.301751056503585, -0.23435547329315712, 1.4911708634625573, 2.33462727784187, 1.8381611752395561, 0.5168426898155446, 0.6052403181611395, -2.3485295719524633, -1.5162261546248468, 2.6256251045200063, 0.04075565307926541], [-1.1417407820116543, 1.7369813007320278, 1.1909698262410706, -1.4879702039550227, 1.8961098797589953, -1.0823287097600482, 0.8963904153006437, 0.7112740908826467, 4.088549763534521, -0.9983194575592643, 1.5022714420788081, 2.7698505689741317, -3.6927788411599365, 0.5402156581188429, 1.5461971818337406, -1.0117970772819525], [-1.4340598309261012, 2.2017570151657133, 1.637790479974621, -1.2956466501431905, 2.1369830522003834, -1.9608327535768706, 1.4896540144769472, 1.276646471857088, 6.196565798803479, -1.2675609855351244, 1.5087053959043026, 2.5639337251960344, -5.580240820476675, 0.16041206434271796, 2.337171595445851, -0.9293062138992693], [-2.1675188704832697, -2.0973315680952553, 2.95992431968253, -1.9526109734370234, 0.6204329963103414, -1.469330014638263, 2.2870331169970184, -1.1534468709480126, -0.4543378858135354, -1.6677130337746964, -0.8597158223798891, -0.6272343306124432, -1.497794976740876, -0.5760490669823534, 2.057045295483614, 2.1504422332199384], [-3.2993365898709515, -2.771243941979489, 2.44340143726662, -7.072161117697205, -1.8116899833713949, -1.4256969412010945, 1.8362167047801898, 0.031317827293346104, 1.3155591510739906, -5.5209301180848245, -0.20956617208139203, 0.026455170705693608, 0.5082809939352079, -2.635803165875005, 3.794782495338149, 3.5910810682427545], [-0.0, 0.0, 0.0, -0.0, 0.0, -0.0, -0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0, 0.0, 0.0, -0.0], [0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, -0.0, 0.0, -0.0, -0.0, -0.0, 0.0, -0.0, -0.0], [0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, -0.0, 0.0, -0.0, -0.0, -0.0, 0.0, -0.0, -0.0]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[0.7168010479838091, 0.5381494606101004, 1.1600447237941225, -1.4682099290383495, 2.333471771947356, -2.1591252813290067, -0.7609885010524969, -1.225108443140735], [0.623320496309205, 0.2654283982345196, 1.4059151986067766, 1.0465642333592868, 0.8449885259727684, -0.01177854040601808, 0.44556608824984967, -0.2592554713823251], [1.158710109073021, 0.7557337767823881, 1.3780167842109998, 0.23515092475280552, -0.08807752995422112, 0.6764244415463714, -0.40202613736545917, -0.17402839548314478], [-0.8502615393925881, 0.6062297184292976, 0.2994660612176008, -0.6167311888885332, 2.1019663302834, 0.6255462170943772, -0.18498627299761086, -0.8770417326111338], [-0.815635492170933, -0.1084405726092009, -1.6476976586039527, -0.8926735030485616, 0.23228446899512464, 0.8833875897279925, -1.1814381153000484, 1.1887305633574106], [0.8214182721190532, -1.0610475676915099, -1.460206218645628, 0.593487795402054, 0.6491745168694203, -0.24742880058087335, 0.7570732282737893, 3.1954814468075052], [-0.019253332981263773, 1.4418971059953938, -0.49562150298009694, -1.4018029553208347, -1.0330875226017409, -1.597386816804304, 1.0642215760851599, 0.25667951136997025]]], "dtype": "float64"}, {"__nd__": [[0.18492359976783937, 0.24204549851988644, -0.2767205955634074, 0.05052841206971315, -0.18666178154326518, -0.14653217444546296, -0.4401045088524067, -0.6369123556735545], [-0.001353739563630954, -0.32735343950117435, -0.2912695377882595, 0.39032917037728126, 0.06300504897463242, -0.008678181211339363, -0.08708403288553444, -0.03909576726042125], [0.4151852607966182, -0.13985981181215384, 0.6876978109536241, 0.574558305286721, -0.5134810584560218, -0.13890863652496446, 0.3764604480435083, -0.31008099296808656], [-0.6527166141174918, 0.11365654632615474, 0.562964928902416, -0.6648150621701939, 0.4224043652460121, -0.5458045920773233, -0.36063303557979515, 0.1598402865746151], [-0.15727377582851532, -0.2350869846537883, -0.17507750648238446, -0.7668842290306485, 0.5624773684220593, -0.202106301772767, -0.451478671065103, -0.49054023784927864], [0.19056978100814329, 0.0892639473767355, -0.7400145600897665, -0.23051660026826382, -0.6169437122651837, 0.12588524416096167, -0.29466213427829463, 0.210978362965722], [0.23192578981984027, 0.4137079870068807, -0.1807151464398421, -0.21590193949325168, 0.04128230465208502, 0.3583823042998817, -0.17421715920456726, -0.20423541514490645], [0.376925926261539, -0.7469651744036141, 0.579605806496809, -0.061823135704800074, 0.12524353639563454, 0.4336048423945488, -0.34670855042804677, 0.09446750671856162]], "dtype": "float64"}, {"__nd__": [[0.4753939412640675, 0.12104029086508623, 0.3058279493066277, -0.2734490695436253, 0.29904487486056497, 0.0834423675373568, -0.1617411679560562, -0.5248774830707779], [0.1763032283034419, -0.4235895351133372, 0.13411495232152476, 0.5901531895217839, -0.14156800598325192, 0.14988236707550193, -0.44220840676089995, 0.31267448042329327], [-0.8361593284345045, 0.001996993383915766, -0.47319970059647387, -0.40142135699662707, -0.396835156408496, 0.6423751130174961, -0.37092024289350023, 0.5665761076593998], [-0.042711497690741305, -0.2821291120933447, -0.028730559896435994, -0.07562311632266257, -0.05650971570716813, 0.4022391231907567, -0.6887070426090338, 0.11182739010813503], [-0.0209086370496963, 0.15262694814278954, -0.3086982224336459, 0.7151498372882266, 0.14454928108153875, -0.3791797032696577, -0.18340144192140287, -0.10748519808565933], [-0.20142067659318597, 0.37233465521668707, -0.4469023396614438, 1.0132231886104834, -0.19038123322905692, -0.07180405887666509, -0.07292872228171175, 0.2317880742794957], [-0.8602932234648922, -0.02331576415420022, 0.2962551458710742, -0.15058797636808263, 0.16931837665172567, 0.7070617125525543, -0.4301440027042072, -0.3395310298026413], [-0.013842734380703749, 0.07083302807241516, -0.031092982377377183, -0.3590946120876886, -0.2922523271280922, -0.20896192131824992, 0.17543271582138154, 1.0181967398258045]], "dtype": "float64"}, {"__nd__": [[0.037455864748161434, -0.6161618189374612, 0.6826696218967238, -0.1617649555251598, 0.31836446777376565, 0.1773381419708387, -0.03175461072011375, 0.05102237773902212], [-0.8613279036550762, 0.41811847766389926, -1.0042040830249916, 0.5931461605606755, -0.04608221891932683, -0.7865751677676509, 0.14169902019770841, 0.7184984000033183], [0.6967337597736121, 0.23325661686266239, 0.2563039949937735, 0.13005687142452071, 0.38388820240026666, -0.07026419868580515, 0.30806160019053414, -0.4632711609118439], [0.19468384918414197, -0.5147777029272437, -0.43986233612371717, 0.5875361914784468, 0.12305021844810966, 0.826616034529478, -0.38814220948387507, -0.7358873015570245], [-0.4073101023597763, 0.3905572344608508, 0.30910281710783655, -0.565276806198617, -0.21527753800226176, 0.06144305038575229, -0.1770090642226575, -0.8666596273723516], [1.073179479391286, 0.8692620881810564, 0.3464213564647929, -0.4141285458012086, -0.7154839600086995, 0.32570354255929945, 0.24110404651004283, 0.47033664820436744], [0.5377393889213921, -0.10311271916391523, -0.12075338297493077, 0.07025479402877365, -0.10521443968900375, -0.4780106162281114, 0.003810775890235068, 0.533700927475744], [0.03328788653515461, 0.1423361568527124, 0.7958004519939467, -0.23707715582876188, 0.00976207370764847, -0.09030594254022717, -0.007276213595533825, 0.15984695594197437]], "dtype": "float64"}, {"__nd__": [[-0.4114272508978338, -0.05028505089206582, 0.9368249092167051, -0.42803253204460207, 0.13139572937361696, 0.5935697605261757, 0.8671843605589183, 0.35579284554400176], [-0.3956999782774861, -0.10559493046179363, 0.5549722663155598, 0.5421591725660329, -0.21633856440533267, -0.007495899539644618, 0.5932561544656814, -0.4111763939790378], [0.6357298841729124, -0.7570926073609706, -0.4401360565446523, 0.2469355586216404, 0.20992807588641132, -0.43045916659290945, 0.763791481301515, -0.32209948959197704], [0.16819506647765678, 0.7786085404310623, 0.2319365813579861, 0.5061266565675603, -0.46472609987861513, -0.21191794238883988, -0.5188679820902898, 0.10935992280800651], [-0.0090180411013162, -0.28780428484923354, 0.27478288976924914, -0.18238952144277082, 0.5205447846612107, 0.20682401458894917, -0.20538345234846292, 0.32196161185223005], [0.09713803841643083, 0.6585849592107254, -0.4062492315731593, 0.449976582804575, -0.04395893565951381, -0.3293709921635201, 0.4131934859679414, -0.4069465298466119], [-0.0940314429752992, 0.08873226381336911, 0.32968949371936196, 0.0845437179163865, -0.34442639346053694, 0.18685649437398774, -0.6396369995580891, -0.6637517840569973], [-0.43729906774068417, 0.2525564091538356, -0.3407134136926861, 0.1243340208076279, -0.07106325370679588, -0.2273556098518159, -0.060900401520126116, 0.7413508910527717]], "dtype": "float64"}, 4, 4, {"__nd__": [7], "dtype": "int64"}, 100], "e": {"__nd__": [[[2.4524822939957356, -2.5488280071021903, -1.652396666501533, -0.30911413614868444, 0.9643366532921092, -0.5021032806550474, -3.839155323706311, -2.0858470504039635], [1.4676313976789281, -1.9446952988768356, 0.38569944769533204, -0.8515964431133558, 0.955317489152857, 0.5737135693636874, -1.7733788904781482, -1.1245533529675813], [1.465934875296341, -0.9134018520077156, -0.17635572012016454, 0.06779752043048481, 0.3936935697783709, -0.08519188892668161, -1.259460118744581, -1.6007065604016706], [0.6757017708758695, -1.0944024027396146, 0.7918535610388947, -0.0859968354708367, 0.15749919152991756, 0.3935944365003353, -0.4676585404595398, -1.3071933981145964], [0.40565406183143893, -0.5428963289712874, -0.20520004832938255, 0.4089356107823625, 0.13551844389124063, -0.23561464952169522, -0.3621740983708777, -0.9731684594062473], [2.6491010555069776, -4.085578332463317, -1.7559054173307442, 0.3539731441615481, 1.5311201275663573, -1.1705299943692151, 2.510673862761755, -2.5300312806518908], [-0.31041549960413406, -0.9089301727968897, -0.7029335119238015, -0.21373205742290668, 0.4642191484683842, -0.08423264261880012, -0.8843583513992673, 0.030657880886516328]]], "dtype": "float64"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(causal_gqa, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")